# Phase 2 step 2c — train the whole Group A grid

Trains every config in `dse/grid.py` that needs its own model, and writes one checkpoint per
config for the sweep runner to pick up.

**Why this is a separate notebook from `dwn_jsc_kaggle.ipynb`.** That one reproduces ONE
config -- the paper's `sm` -- and is the Gate 1 reference. It should stay exactly as it is.
This one loops.

**Three things it does that a loop over the Phase 1 notebook would not:**

1. **Groups configs by (encoding, z) and binarizes once per group.** 16 of the 32 configs share
   `(distributive, 200)` -- the whole width ladder, the `n` sweep and the layer-count sweep --
   so binarizing per config would repeat the most expensive setup step 16 times for identical
   output. 32 configs need only **9** binarizations.
2. **Falls back to on-the-fly binarization when the precomputed set will not fit.** At z=800 the
   binarized data is ~10.6 GB as uint8, which does not fit Kaggle RAM alongside everything else.
   Above a threshold, batches are binarized on the GPU instead. The two paths produce identical
   bits -- there is an assertion that checks exactly that -- so this is a speed/memory tradeoff,
   never a correctness one.
3. **Is resumable.** It skips any config whose checkpoint already exists in `/kaggle/working`.
   32 training runs will not fit one session; re-running the notebook continues where it left
   off. Set `ONLY_N` to bound a session deliberately rather than by timeout.

**Filenames are the contract.** Each checkpoint is `<slug>_checkpoint.pt`, where the slug comes
from `ModelConfig.slug` (`n6_z200_distributive_w50`) -- the identity of the *trained model*, with
no hardware parameters in it. `dse/run.py` resolves checkpoints by exactly that name. Rename
these and the sweep silently finds nothing.

Settings: **Accelerator -> GPU**, **Internet -> On**.

In [ ]:
# ---- environment check: fail loudly and early ----
import subprocess, sys, torch

print('torch     :', torch.__version__)
print('cuda avail:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu       :', torch.cuda.get_device_name(0))
    print('cuda (torch built against):', torch.version.cuda)
else:
    raise SystemExit('No GPU. Set Accelerator -> GPU in the settings panel. '
                     'DWN training cannot run on CPU.')

print()
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout)
print('If the nvcc CUDA version and the torch CUDA version differ a lot, the extension')
print('build in the next cell is where it will show up.')

In [ ]:
# ---- clone upstream DWN at the pinned commit and build the CUDA extension ----
# Pin matches third_party/DWN in the repo. Do not float this to main: the exporter is
# built against whatever checkpoint format this commit produces (CLAUDE.md).
PINNED_COMMIT = '9f887a0b4bd84dabf6d8c9ae35368ab2a7e0e3c0'

!rm -rf /kaggle/working/DWN
!git clone --quiet https://github.com/alanbacellar/DWN.git /kaggle/working/DWN
!cd /kaggle/working/DWN && git checkout --quiet {PINNED_COMMIT} && git log -1 --format='pinned at %h %ad %s'

# Confirm the CUDA sources actually exist at this pin before spending 5 minutes on a build.
# The pinned commit is literally "Delete custom_operators/cuda directory (Duplicate)" -- it
# removed a duplicate copy, not the real one, but that is worth verifying rather than assuming.
!ls -la /kaggle/working/DWN/src/torch_dwn/custom_operators/cuda/

# This compiles efd_cuda_kernel.cu with nvcc. Expect 2-5 minutes. It is the slowest and
# most fragile step in the notebook.
#
# --no-build-isolation is REQUIRED, not an optimization. Upstream's pyproject.toml declares
#     [build-system] requires = ["setuptools>=42", "wheel", "torch"]
# so a plain `pip install .` builds in a fresh isolated env and downloads ANOTHER torch from
# PyPI. setup.py's `import torch` then resolves to that one instead of the session's, so the
# extension gets built against a torch/CUDA pair that does not match the runtime -- it either
# fails to compile outright or builds and then fails to import on an ABI mismatch.
#
# Full output on purpose. Do NOT pipe this through `tail`: real compiler errors appear near
# the TOP of the log, while the last 20 lines are always the same generic pip epilogue
# ("did not run successfully / See above for output"), which identifies nothing.
!cd /kaggle/working/DWN && pip install --no-build-isolation .


In [ ]:
# ---- VERIFY the extension actually built ----
# This cell exists because the failure is otherwise silent. lut_layer.py does
#     if torch.cuda.is_available(): import efd_cuda
# so a failed build produces no error at install time -- you would instead get a bare
# NameError at the first forward pass, long after the real cause.
import torch, torch_dwn as dwn

try:
    import efd_cuda
    print('efd_cuda imported OK')
except ImportError as e:
    raise SystemExit(
        'efd_cuda failed to import -- the CUDA extension did not build.\n'
        'Scroll to the TOP of the install cell output and read the first compiler error;\n'
        'the tail of a pip failure is generic boilerplate and never names the cause.\n'
        f'Original error: {e}'
    )

# tiny end-to-end forward+backward, so we find out here rather than 200 lines later
_probe = torch.nn.Sequential(dwn.LUTLayer(12, 6, n=6), dwn.GroupSum(k=2, tau=1.0)).cuda()
_x = (torch.rand(4, 12, device='cuda') > 0.5).float()
_out = _probe(_x)
_out.sum().backward()
print('forward + backward OK, output shape', tuple(_out.shape))
del _probe, _x, _out


In [ ]:
# ---- the grid: one source of truth ----
# Generated by `python dse/grid.py --json`. Embedded so the notebook runs standalone, but it
# prefers a copy uploaded as a Kaggle dataset -- regenerate and re-upload after changing the
# grid, or the models trained here stop matching the configs synthesized at home.
import glob, json

EMBEDDED_GRID = {"training_set": [{"slug": "n6_z200_distributive_w50","label": "1x50","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [50],"mapping": ["learnable"],"num_classes": 5,"tau": 3.333333333333334,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w100","label": "1x100","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [100],"mapping": ["learnable"],"num_classes": 5,"tau": 4.902385484305461,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w200","label": "1x200","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w360","label": "1x360","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w500","label": "1x500","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [500],"mapping": ["learnable"],"num_classes": 5,"tau": 12.318032472459624,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w600","label": "1x600","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [600],"mapping": ["learnable"],"num_classes": 5,"tau": 13.829048091198098,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w800","label": "1x800","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [800],"mapping": ["learnable"],"num_classes": 5,"tau": 16.599017813493987,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w1200","label": "1x1200","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [1200],"mapping": ["learnable"],"num_classes": 5,"tau": 21.47017162732062,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w1600","label": "1x1600","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [1600],"mapping": ["learnable"],"num_classes": 5,"tau": 25.770664687144944,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w2000","label": "1x2000","group": "ladder","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [2000],"mapping": ["learnable"],"num_classes": 5,"tau": 29.691203596049384,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z8_distributive_w200","label": "1x200 z=8","group": "ofat-z","n": 6,"thermometer_bits": 8,"thermometer": "distributive","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z25_distributive_w200","label": "1x200 z=25","group": "ofat-z","n": 6,"thermometer_bits": 25,"thermometer": "distributive","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z50_distributive_w200","label": "1x200 z=50","group": "ofat-z","n": 6,"thermometer_bits": 50,"thermometer": "distributive","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z100_distributive_w200","label": "1x200 z=100","group": "ofat-z","n": 6,"thermometer_bits": 100,"thermometer": "distributive","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z400_distributive_w200","label": "1x200 z=400","group": "ofat-z","n": 6,"thermometer_bits": 400,"thermometer": "distributive","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z800_distributive_w200","label": "1x200 z=800","group": "ofat-z","n": 6,"thermometer_bits": 800,"thermometer": "distributive","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_gaussian_w200","label": "1x200 gaussian","group": "ofat-enc","n": 6,"thermometer_bits": 200,"thermometer": "gaussian","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_linear_w200","label": "1x200 linear","group": "ofat-enc","n": 6,"thermometer_bits": 200,"thermometer": "linear","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n4_z200_distributive_w200","label": "1x200 n=4","group": "ofat-n","n": 4,"thermometer_bits": 200,"thermometer": "distributive","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n2_z200_distributive_w200","label": "1x200 n=2","group": "ofat-n","n": 2,"thermometer_bits": 200,"thermometer": "distributive","layers": [200],"mapping": ["learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w100x100","label": "2x100","group": "ofat-L","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [100,100],"mapping": ["learnable","learnable"],"num_classes": 5,"tau": 7.2100150310186635,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w65x65x65","label": "3x65","group": "ofat-L","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [65,65,65],"mapping": ["learnable","learnable","learnable"],"num_classes": 5,"tau": 7.10913951343757,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z8_distributive_w360","label": "1x360 z=8","group": "ofat-z","n": 6,"thermometer_bits": 8,"thermometer": "distributive","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z25_distributive_w360","label": "1x360 z=25","group": "ofat-z","n": 6,"thermometer_bits": 25,"thermometer": "distributive","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z50_distributive_w360","label": "1x360 z=50","group": "ofat-z","n": 6,"thermometer_bits": 50,"thermometer": "distributive","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z100_distributive_w360","label": "1x360 z=100","group": "ofat-z","n": 6,"thermometer_bits": 100,"thermometer": "distributive","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z400_distributive_w360","label": "1x360 z=400","group": "ofat-z","n": 6,"thermometer_bits": 400,"thermometer": "distributive","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z800_distributive_w360","label": "1x360 z=800","group": "ofat-z","n": 6,"thermometer_bits": 800,"thermometer": "distributive","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_gaussian_w360","label": "1x360 gaussian","group": "ofat-enc","n": 6,"thermometer_bits": 200,"thermometer": "gaussian","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_linear_w360","label": "1x360 linear","group": "ofat-enc","n": 6,"thermometer_bits": 200,"thermometer": "linear","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n4_z200_distributive_w360","label": "1x360 n=4","group": "ofat-n","n": 4,"thermometer_bits": 200,"thermometer": "distributive","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n2_z200_distributive_w360","label": "1x360 n=2","group": "ofat-n","n": 2,"thermometer_bits": 200,"thermometer": "distributive","layers": [360],"mapping": ["learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w180x180","label": "2x180","group": "ofat-L","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [180,180],"mapping": ["learnable","learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802},{"slug": "n6_z200_distributive_w120x120x120","label": "3x120","group": "ofat-L","n": 6,"thermometer_bits": 200,"thermometer": "distributive","layers": [120,120,120],"mapping": ["learnable","learnable","learnable"],"num_classes": 5,"tau": 10.000000000000002,"batch_size": 100,"epochs": 32,"lr": 0.01,"lr_step": 14,"lr_gamma": 0.1,"seed": 20260802}],"count": 34}

found = glob.glob('/kaggle/input/**/train_grid.json', recursive=True)
if found:
    GRID = json.load(open(found[0]))
    print(f'grid: {found[0]} (uploaded)')
else:
    GRID = EMBEDDED_GRID
    print('grid: embedded copy (no train_grid.json in /kaggle/input)')

TRAINING_SET = GRID['training_set']
print(f'{len(TRAINING_SET)} configs need training')

# Bound a session deliberately. None = attempt all; a timeout mid-run is safe (resumable)
# but wastes whatever the current config had done.
ONLY_N = None

# Split the grid across accounts or sessions. None = no filter.
#
# Work partitions cleanly by (encoding, z) because that is the unit a binarization is shared
# over -- so two machines splitting on z duplicate NO setup work, and their outputs cannot
# collide, since checkpoint filenames are slug-based and a differing z means a differing slug.
#
# The z=200 block is 20 of the 32 configs and about 6 hours; z=400 and z=800 are only 4 configs
# but nearly 3 hours, because cost scales with z rather than with model size. Splitting the tail
# off is therefore the cheapest way to fit both halves inside a session cap:
#
#     machine A   ONLY_Z = None          (or leave it -- it will skip what B has done)
#     machine B   ONLY_Z = (400, 800)
#
# Both machines MUST run the same grid. Mixing a pre-2026-08-08 notebook with this one would
# mix two tau schedules across the width ladder, which is the bug that forced the restart.
ONLY_Z = None
ONLY_ENCODING = None       # e.g. ('gaussian', 'linear')

# Exact slugs, for picking up stragglers. The most precise filter, and the one to use when a
# session died or a download missed a file: it needs no input dataset, because there is nothing
# to skip -- you name only what is absent.
#     ONLY_SLUGS = ('n6_z200_distributive_w800', 'n6_z200_linear_w360')
ONLY_SLUGS = None

# Above this, the precomputed binarized set is skipped and batches are binarized on the GPU.
# 664k x 16 x z bytes; z=800 is ~10.6 GB, which does not fit alongside everything else.
PRECOMPUTE_LIMIT_GB = 6.0

if ONLY_SLUGS:
    TRAINING_SET = [c for c in TRAINING_SET if c['slug'] in ONLY_SLUGS]
    got = {c['slug'] for c in TRAINING_SET}
    for s in ONLY_SLUGS:
        if s not in got:
            print(f'WARNING: {s!r} is not a slug in this grid -- typo? It will NOT be trained.')
if ONLY_Z:
    TRAINING_SET = [c for c in TRAINING_SET if c['thermometer_bits'] in ONLY_Z]
if ONLY_ENCODING:
    TRAINING_SET = [c for c in TRAINING_SET if c['thermometer'] in ONLY_ENCODING]
if ONLY_Z or ONLY_ENCODING or ONLY_SLUGS:
    print(f'FILTERED to {len(TRAINING_SET)} configs '
          f'(slugs={"set" if ONLY_SLUGS else "any"}, z={ONLY_Z or "any"}, '
          f'encoding={ONLY_ENCODING or "any"})')
    for c in TRAINING_SET:
        print(f'    {c["slug"]}')
    print('Another machine is expected to cover the rest -- merge the outputs when both finish.')

import collections
groups = collections.Counter((c['thermometer'], c['thermometer_bits']) for c in TRAINING_SET)
print(f'{len(groups)} distinct (encoding, z) binarizations for {len(TRAINING_SET)} configs:')
for (kind, z), n in sorted(groups.items(), key=lambda kv: -kv[1]):
    gb = 830000 * 16 * z / 1e9
    note = '  -> on-the-fly (too big to precompute)' if gb > PRECOMPUTE_LIMIT_GB else ''
    print(f'  {kind:14s} z={z:4d}  {n:2d} configs  ~{gb:5.2f} GB{note}')

In [ ]:
# ---- load JSC ----
# Identical to the Phase 1 notebook: same split, same seed, same scaler. That is deliberate --
# a sweep whose data pipeline differs from the reference config's cannot be compared to it.
import numpy as np, torch
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

SEED = TRAINING_SET[0]['seed']
torch.manual_seed(SEED)
np.random.seed(SEED)

data = fetch_openml('hls4ml_lhc_jets_hlf', version=1, as_frame=True)
X = data.data.to_numpy(dtype=np.float32)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data.target.to_numpy())
assert X.shape[1] == 16 and len(label_encoder.classes_) == 5

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)
y_train_t = torch.from_numpy(y_train).long()
y_test_t = torch.from_numpy(y_test).long()
print('train', tuple(X_train_t.shape), ' test', tuple(X_test_t.shape))
print('classes:', list(label_encoder.classes_), '(alphabetical -- index 2 is top, not W)')

In [ ]:
# ---- helpers ----
import time, os, gc
import torch_dwn as dwn
from torch import nn
from torch.nn.functional import cross_entropy

THERMOMETERS = {
    'plain': dwn.Thermometer,
    'linear': dwn.Thermometer,            # grid calls evenly-spaced 'linear'
    'gaussian': dwn.GaussianThermometer,
    'distributive': dwn.DistributiveThermometer,
}
WORK = '/kaggle/working'


def binarize_chunked(therm, x, chunk=10000):
    """Binarize -> flatten -> uint8 in slices, to bound peak memory (Phase 1 cell 6)."""
    out = torch.empty((x.size(0), x.size(1) * therm.num_bits), dtype=torch.uint8)
    for i in range(0, x.size(0), chunk):
        out[i:i + chunk] = therm.binarize(x[i:i + chunk]).flatten(start_dim=1).to(torch.uint8)
    return out


def make_group(kind, z):
    """Fit the thermometer and prepare batch access for every config sharing (kind, z)."""
    therm = THERMOMETERS[kind](z).fit(X_train_t)
    gb = (X_train_t.size(0) + X_test_t.size(0)) * 16 * z / 1e9
    thr_gpu = therm.thresholds.cuda()

    def on_the_fly(xf):
        # Same comparison binarization.py performs, done on the GPU so no huge CPU tensor
        # is ever materialized.
        return (xf.cuda().unsqueeze(-1) > thr_gpu).flatten(start_dim=1).float()

    if gb <= PRECOMPUTE_LIMIT_GB:
        xb_tr, xb_te = binarize_chunked(therm, X_train_t), binarize_chunked(therm, X_test_t)
        # The two paths MUST agree, or a large-z config would be trained on different bits
        # than a small-z one -- a difference that would look like a result.
        probe = on_the_fly(X_test_t[:256]).to(torch.uint8).cpu()
        assert torch.equal(probe, xb_te[:256]), 'on-the-fly binarization != precomputed'
        print(f'    precomputed {gb:.2f} GB uint8 (GPU path verified identical on 256 samples)')
        return therm, (lambda i: xb_tr[i].cuda().float()), (lambda i: xb_te[i].cuda().float()), \
            xb_tr.size(1), (xb_tr, xb_te)

    print(f'    on-the-fly binarization ({gb:.2f} GB would not fit)')
    return therm, (lambda i: on_the_fly(X_train_t[i])), (lambda i: on_the_fly(X_test_t[i])), \
        16 * z, None


def train_one(cfg, get_train, get_test, in_bits):
    """Train one config. Returns (model, results dict)."""
    torch.manual_seed(cfg['seed'])
    layers, size = [], in_bits
    for i, w in enumerate(cfg['layers']):
        layers.append(dwn.LUTLayer(size, w, n=cfg['n'], mapping=cfg['mapping'][i]))
        size = w
    layers.append(dwn.GroupSum(k=cfg['num_classes'], tau=cfg['tau']))
    model = nn.Sequential(*layers).cuda()

    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    sched = torch.optim.lr_scheduler.StepLR(opt, cfg['lr_step'], cfg['lr_gamma'])
    n_train, n_test = X_train_t.size(0), X_test_t.size(0)

    def evaluate(chunk=5000):
        model.eval()
        correct = 0
        with torch.no_grad():
            for i in range(0, n_test, chunk):
                sl = slice(i, i + chunk)
                correct += (model(get_test(sl)).argmax(1)
                            == y_test_t[sl].cuda()).sum().item()
        return correct / n_test

    best, history, losses = 0.0, [], []
    t0 = time.time()
    for ep in range(cfg['epochs']):
        model.train()
        perm = torch.randperm(n_train)
        run, nb = 0.0, 0
        for i in range(0, n_train, cfg['batch_size']):
            idx = perm[i:i + cfg['batch_size']]
            opt.zero_grad()
            loss = cross_entropy(model(get_train(idx)), y_train_t[idx].cuda())
            loss.backward()
            opt.step()
            run += loss.item(); nb += 1
        sched.step()
        losses.append(run / nb)
        acc = evaluate()
        history.append(acc); best = max(best, acc)
        if ep % 8 == 7 or ep == cfg['epochs'] - 1:
            print(f'      epoch {ep+1:3d}/{cfg["epochs"]}  loss {losses[-1]:.4f}  '
                  f'acc {acc:.4f}  [{time.time()-t0:.0f}s]')
    return model, {'final_acc': history[-1], 'best_acc': best,
                   'history': history, 'epoch_losses': losses,
                   'seconds': round(time.time() - t0, 1)}


def save_config(cfg, model, therm, res, get_test):
    """Checkpoint + 1000-sample test vectors, in the format exporter/extract.py reads."""
    slug = cfg['slug']
    ck_path = f'{WORK}/{slug}_checkpoint.pt'
    torch.save({
        'run_name': slug,
        # Exactly the keys the Phase 1 checkpoint carries -- extract.py reads config['n'],
        # ['layers'], ['thermometer_bits'], ['num_classes'] (docs/checkpoint-format.md).
        'config': {k: cfg[k] for k in (
            'thermometer', 'thermometer_bits', 'n', 'layers', 'mapping', 'num_classes',
            'tau', 'batch_size', 'epochs', 'lr', 'lr_step', 'lr_gamma', 'seed')},
        'pinned_commit': PINNED_COMMIT,
        'state_dict': model.state_dict(),
        'thermometer': {'kind': cfg['thermometer'], 'num_bits': cfg['thermometer_bits'],
                        'thresholds': therm.thresholds.cpu()},
        'scaler': {'mean': torch.from_numpy(scaler.mean_.astype(np.float32)),
                   'scale': torch.from_numpy(scaler.scale_.astype(np.float32))},
        'classes': list(label_encoder.classes_),
        'feature_names': list(data.feature_names),
        'results': res,
        'grid_label': cfg['label'],
        'torch_version': torch.__version__,
    }, ck_path)

    # Gate 1 needs these: gen_vectors.py reads <run>_testvectors.npz beside the checkpoint.
    # x_binarized drives the core testbench, x_raw the encoder+core one -- both, so a Gate 1
    # failure localizes to one or the other.
    model.eval()
    with torch.no_grad():
        xb = get_test(slice(0, 1000))
        pred = model(xb).argmax(1).cpu().numpy()
    np.savez_compressed(
        f'{WORK}/{slug}_testvectors.npz',
        x_binarized=xb.to(torch.uint8).cpu().numpy(),
        x_raw=X_test[:1000], y=y_test[:1000], pred=pred)
    return ck_path

In [ ]:
# ---- carry forward anything already trained in an EARLIER SESSION ----
# /kaggle/working is fresh on every version -- it only persists while one session is alive.
# So the resume check below would restart from zero on a new session, which is precisely the
# case a 32-config run needs. Add the previous run's OUTPUT as an input dataset and this copies
# it forward, so the final Output panel holds the complete set rather than one session's slice.
import glob, shutil

carried = 0
for src in glob.glob('/kaggle/input/**/*_checkpoint.pt', recursive=True) + \
           glob.glob('/kaggle/input/**/*_testvectors.npz', recursive=True):
    dst = os.path.join(WORK, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        carried += 1
print(f'carried forward {carried} file(s) from /kaggle/input')
if not carried:
    print('(none -- first session, or no previous output added as an input dataset)')

In [ ]:
# ---- train the grid ----
# Ordered by (encoding, z) so each binarization is built once and reused by every config that
# shares it. Within a group, configs run cheapest-first so a session that dies late still
# banks the most models.
import collections

by_group = collections.defaultdict(list)
for c in TRAINING_SET:
    by_group[(c['thermometer'], c['thermometer_bits'])].append(c)
for v in by_group.values():
    v.sort(key=lambda c: sum(c['layers']))

done = [c for c in TRAINING_SET if os.path.exists(f"{WORK}/{c['slug']}_checkpoint.pt")]
todo = [c for c in TRAINING_SET if c not in done]
print(f'{len(done)} already trained, {len(todo)} to go')
if ONLY_N:
    print(f'ONLY_N={ONLY_N}: stopping after {ONLY_N} this session')

summary, trained = [], 0
for (kind, z), configs in sorted(by_group.items(), key=lambda kv: kv[0][1]):
    pending = [c for c in configs
               if not os.path.exists(f"{WORK}/{c['slug']}_checkpoint.pt")]
    if not pending or (ONLY_N and trained >= ONLY_N):
        continue
    print(f'\n=== {kind}, z={z} -- {len(pending)} config(s) ===')
    therm, get_train, get_test, in_bits, held = make_group(kind, z)

    for cfg in pending:
        if ONLY_N and trained >= ONLY_N:
            break
        print(f'  -- {cfg["slug"]}  ({cfg["label"]}, {sum(cfg["layers"])} nodes)')
        model, res = train_one(cfg, get_train, get_test, in_bits)
        path = save_config(cfg, model, therm, res, get_test)
        print(f'     final {res["final_acc"]:.4f}  best {res["best_acc"]:.4f}  '
              f'{res["seconds"]:.0f}s  -> {os.path.basename(path)}')
        summary.append((cfg['label'], cfg['slug'], res['final_acc'], res['seconds']))
        trained += 1
        del model; gc.collect(); torch.cuda.empty_cache()

    del therm, get_train, get_test, held
    gc.collect(); torch.cuda.empty_cache()

print(f'\ntrained {trained} config(s) this session')

In [ ]:
# ---- summary ----
import glob
cks = sorted(glob.glob(f'{WORK}/*_checkpoint.pt'))
print(f'{len(cks)} checkpoints in {WORK} ({len(TRAINING_SET)} configs in the grid)')
print()
if summary:
    print(f'{"config":24s} {"final acc":>10} {"seconds":>9}')
    print('-' * 46)
    for label, slug, acc, secs in summary:
        print(f'{label:24s} {100*acc:>9.2f}% {secs:>9.0f}')
    print('-' * 46)

missing = [c['slug'] for c in TRAINING_SET
           if not os.path.exists(f"{WORK}/{c['slug']}_checkpoint.pt")]
if missing:
    print(f'\nSTILL MISSING ({len(missing)}): re-run this notebook to continue.')
    for s in missing[:10]:
        print('  ', s)
    if len(missing) > 10:
        print(f'   ... and {len(missing)-10} more')
else:
    print('\nAll configs trained. Download every *_checkpoint.pt AND *_testvectors.npz')
    print('into training/artifacts/, then run:  python dse/run.py --all --impl')

## After this runs

Download **both** files per config from the Output panel into `training/artifacts/`:

- `<slug>_checkpoint.pt` — what the exporter reads
- `<slug>_testvectors.npz` — what Gate 1 simulates against

`dse/run.py` resolves checkpoints by slug, so the filenames must not be changed. Then:

```
python dse/run.py --list          # confirms which configs now have checkpoints
python dse/run.py --all --impl    # Gate 1 + place-and-route for every config
python dse/report.py              # table, frontier, headline number
python dse/plot.py                # figures
```

Use `--impl`: post-synthesis timing uses estimated routing and is systematically optimistic —
Phase 1 measured 161.0 MHz post-synthesis against 147.1 MHz post-route on the same design.

**This notebook does not produce a full 166k test-set dump per config.** Gate 1b — reproducing
software accuracy on real silicon over the whole test set — is Phase 1's exit condition and was
done once, for the reference config. Phase 2 measures area and timing from Vivado reports, and
takes accuracy from the checkpoint, so a 9.4 MB dump per config would be 300 MB of files nothing
reads. If a specific sweep config is ever taken to hardware, run `dump_testset_kaggle.ipynb`
against that one checkpoint.